# Project 1: Chat with your PDF (RAG)
**Tech:** Python, sentence-transformers, FAISS, Groq LLM (free API), Gradio

- Built a Retrieval-Augmented Generation app that answers questions from user-uploaded PDFs using FAISS vector search and an LLM.
- Tested on [X] documents / [Y] questions; average response time [Z] seconds; answers cite source page numbers.

### Cell 1: Libraries install karo

In [8]:
!pip install -q pypdf sentence-transformers faiss-cpu groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.5 MB/s eta 0:00:00


### Cell 2: PDF upload and (file picker)

In [36]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Loaded:", pdf_path)

Saving Internship Report_merged.pdf to Internship Report_merged.pdf
Loaded: Internship Report_merged.pdf


### Cell 3: picks text of pdf and making chunks of txt into pdf

In [37]:
from pypdf import PdfReader

def load_pdf(path):
    reader = PdfReader(path)
    return [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]

def chunk_text(pages, size=800, overlap=150):
    chunks = []
    for pg, text in pages:
        text = " ".join(text.split())
        start = 0
        while start < len(text):
            chunks.append({"page": pg, "text": text[start:start + size]})
            start += size - overlap
    return chunks

pages = load_pdf(pdf_path)
chunks = chunk_text(pages)
print(len(pages), "pages ->", len(chunks), "chunks")

23 pages -> 36 chunks


### Cell 4: Making Embeddings and enter the  FAISS index

In [38]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
emb = embedder.encode([c["text"] for c in chunks], convert_to_numpy=True,
                      normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb.astype("float32"))
print("Index ready:", index.ntotal, "vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index ready: 36 vectors


### Cell 5: Groq API key

In [30]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-20b"

Check model name lists

In [39]:
for m in client.models.list().data:
  print(m.id)

whisper-large-v3
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-20b


### Cell 6: Retrieve + Answer function (core RAG logic)

In [40]:
import time

def retrieve(question, k=4):
    qv = embedder.encode([question], convert_to_numpy=True,
                         normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return [chunks[i] for i in ids[0]]

def answer(question):
    start = time.time()
    ctx = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in ctx)
    messages = [
        {"role": "system", "content": "Answer ONLY using the given context. "
         "If the answer is not in the context, say you could not find it in the document."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.2)
    pages_used = sorted({c["page"] for c in ctx})
    return resp.choices[0].message.content, pages_used, time.time() - start

print(answer("Give a short summary of this document."))

('The document is an internship report written by Ms.\u202fShraddha\u202fDeepak\u202fLokhande for her B.Tech final year at SITCOE, Yadrav. It acknowledges the support of the internship coordinator (Mrs.\u202fP.P.\u202fMane), the project guide (Mrs.\u202fV.A.\u202fKole), the department head (Dr.\u202fS.S.\u202fGurav), the principal (Dr.\u202fB.B.\u202fSangame), and the company Paarsh Infotech Pvt.\u202fLtd. The report was submitted under the guidance of Dr.\u202fBabasaheb Ambedkar Technological University and meets the requirements for the Bachelor of Technology degree.', [2, 3], 0.6409156322479248)


### Cell 7: Chat UI ( showing link for using )

In [41]:
import gradio as gr

def chat_fn(message, history):
    ans, pgs, t = answer(message)
    return f"{ans}\n\n(Pages: {pgs} | {t:.1f}s)"

gr.ChatInterface(chat_fn, title="Chat with your PDF").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8bfd450df80b4f2de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Cell 8: Resume metrics

In [44]:
test_questions = [
    "Q.1. tell me skills list",
    "Q.2. project objective"
    "Q.3. Plan Of Internship Program",
]
times = []
for q in test_questions:
    ans, pgs, t = answer(q)
    times.append(t)
    print("Q:", q, "\nA:", ans[:200], "\nPages:", pgs, f"| {t:.1f}s\n")
print(f"Average response time: {sum(times)/len(times):.1f}s over {len(times)} questions")

Q: Q.1. tell me skills list 
A: **Skills list (as mentioned in the document)**  

- **Frontend Development**  
  - HTML5  
  - CSS3  
  - Bootstrap  
  - Responsive Web Design  
  - CSS Flexbox  
  - CSS Grid  
  - Media Queries  

 
Pages: [6, 7, 19, 21] | 1.1s

Q: Q.2. project objectiveQ.3. Plan Of Internship Program 
A: **Q.2. Project Objective**  
I could not find the project objective in the provided document.

**Q.3. Plan Of Internship Program**  
The internship program was organized into several distinct phases t 
Pages: [3, 4, 5, 6] | 0.5s

Average response time: 0.8s over 2 questions
